In [0]:

from datetime import date, timedelta
from pyspark.sql.functions import *

yesterday = date.today() - timedelta(days=1)
yesterday_str = yesterday.strftime("%Y%m%d")

dbutils.widgets.text("file_run_date", yesterday_str)

file_run_date = dbutils.widgets.get("file_run_date")
print(file_run_date)


# dbutils.widgets.text("file_run_date", "20260207")
# file_run_date = dbutils.widgets.get("file_run_date")

In [0]:
%skip
delete from `spark-databricks-project-catalog`.gold.customers;
delete from `spark-databricks-project-catalog`.gold.products;
delete from `spark-databricks-project-catalog`.gold.fact_orders;

## Customers - Dimension Table

In [0]:
%sql
create table if not exists `spark-databricks-project-catalog`.gold.customers --if table exists in rerun case, this    will be skipped
using delta
as 
select sha2(mobile_number, 256) as customer_sk, --surrogate key (sk), keep it in fact table too
       full_name,
       email,
       mobile_number,
       city,
       state,
       zipcode,
       signup_date,
       offline_customer_id,
       online_customer_id,
       file_date,
       source,
       ingest_ts,
       current_timestamp() as last_updated_ts --new column as we need SCD type 1 data .ie. only latest
 from `spark-databricks-project-catalog`.silver.customers_unified
 where 1 = 2;  

In [0]:
%sql
----SCD Type 1

merge into `spark-databricks-project-catalog`.gold.customers t
using (
    select * from (
        select sha2(mobile_number, 256) as customer_sk,
           full_name,
           email,
           mobile_number,
           city,
           state,
           zipcode,
           signup_date,
           offline_customer_id,
           online_customer_id,
           file_date,
           source,
           ingest_ts,
           row_number() over (partition by mobile_number order by ingest_ts desc) as rn
         from `spark-databricks-project-catalog`.silver.customers_unified
         where file_date = :file_run_date
    )
    where rn = 1
) s
on t.mobile_number = s.mobile_number

--if same mobile no.(customer) is appearing again
when matched then update set  
  t.customer_sk = s.customer_sk,
  t.full_name = s.full_name,
  t.email = s.email,
  t.city = s.city,
  t.state = s.state,
  t.zipcode = s.zipcode,
  t.signup_date = s.signup_date,
  t.offline_customer_id = s.offline_customer_id,
  t.online_customer_id = s.online_customer_id,
  t.file_date = s.file_date,
  t.source = s.source,
  t.ingest_ts = s.ingest_ts,
  t.last_updated_ts = current_timestamp()

when not matched then insert (
  customer_sk,
  full_name,
  email,
  mobile_number,
  city,
  state,
  zipcode,
  signup_date,
  offline_customer_id,
  online_customer_id,
  file_date,
  source,
  ingest_ts,
  last_updated_ts
)
values (
   s.customer_sk,
   s.full_name,
   s.email,
   s.mobile_number,
   s.city,
   s.state,
   s.zipcode,
   s.signup_date,
   s.offline_customer_id,
   s.online_customer_id,
   s.file_date,
   s.source,
   s.ingest_ts,
   current_timestamp()
);


## Products Dimension Table

In [0]:
%sql
create table if not exists `spark-databricks-project-catalog`.gold.products
( 
    product_sk string,
    sku_id string,
    product_name string,
    category string,
    brand string,
    price float,
    currency string,
    is_active string,
    tax_rate float,
    file_date date,
    source string,
    ingest_ts timestamp,
    effective_from date, -- SCD Type 2
    effective_to date,  -- SCD Type 2
    is_current boolean --- if current record then 1 else 0
)
using delta;  ---not requried as by default it creates a Delta table

In [0]:
%sql

-- sku_id , price , file_run_date , efeective_from , expire_date
-- sku1 , 100 , 20260101 , 1-jan, 10-jan
-- sku1 , 110 , 20260111 , 11jan , 9999-12-31

-- new load/change happening on another day -> sku1 , 110 , 20260111 - need to expire the old/existing record and maintain new record with latest info
-- sku2 , 110 , 20260111 - here only add as new sku_id has appeared

-- creating a view so that the change once captured (change_hash), could be use while update & insert
CREATE OR REPLACE TEMP VIEW v_products_run AS
SELECT
    sku_id,
    product_name,
    category,
    brand,
    price,
    tax_rate,
    currency,
    is_active,
    file_date,
    source,
    ingest_ts,

    sha2(
        concat_ws('||', sku_id, file_date),
        256
    ) AS product_sk, -- surrogate key based on sku_id & date, as sku_id can repeat while maintaining history, hence to keep it unique surrogate key is a combination of sku_id & file_date

    sha2(
        concat_ws(
            '||',
            product_name,
            category,
            brand,
            price,
            tax_rate,
            currency,
            is_active
        ),
        256
    ) AS change_hash -- to identify if there is any change in the product details

FROM `spark-databricks-project-catalog`.silver.products_unified

WHERE file_date = :file_run_date; -- for current date, created a temp view

In [0]:
%sql
DESCRIBE v_products_run;

In [0]:
%sql

-- Implementing SCD Type 2


merge into `spark-databricks-project-catalog`.gold.products t
using (
    select * from (
        select *,
            row_number() over (partition by sku_id order by ingest_ts desc) as rn
        from v_products_run
    )
    where rn = 1  --only taking latest sku_id record 
) s
on t.sku_id = s.sku_id and 
   t.is_current = True -- whatever records r coming again from source, on the basis of sku_id & is_current = True, and if there is a change (catching by below hash ) then we will expire (update) the old record

when matched and 
   sha2(concat_ws('||', 
                     t.product_name,
                     t.category,
                     t.brand,
                     t.price,
                     t.currency,
                     t.is_active,
                     t.tax_rate), 256) <> s.change_hash --again we're craeting a change hash in target & if there is a change then we will update the target details as below
then update set
    t.is_current = False,
    t.effective_to = date_sub( to_date(:file_run_date, 'yyyyMMdd'), 1)

In [0]:
%sql

-- Update done above, now Insert whatever is new and update too

INSERT INTO `spark-databricks-project-catalog`.gold.products
SELECT
    s.product_sk,
    s.sku_id,
    s.product_name,
    s.category,
    s.brand,
    s.price,
    s.currency,
    s.is_active,
    s.tax_rate,
    to_date(s.file_date, 'yyyyMMdd') AS file_date,
    s.source,
    s.ingest_ts,
    to_date(CAST(s.file_date AS STRING), 'yyyyMMdd') AS effective_from,
    TIMESTAMP '9999-12-31' AS effective_to,
    TRUE AS is_current
FROM v_products_run s --has data from silver.products_unified

LEFT JOIN `spark-databricks-project-catalog`.gold.products t
    ON s.sku_id = t.sku_id
    AND t.is_current = TRUE

WHERE t.sku_id IS NULL
   OR sha2(concat_ws(
                   '||',
                   t.product_name,
                   t.category,
                   t.brand,
                   t.price,
                   t.currency,
                   t.is_active,
                   t.tax_rate
                 ), 256) <> s.change_hash;

## Order's Fact Table

In [0]:
%sql
create table if not exists `spark-databricks-project-catalog`.gold.fact_orders 
(
    order_id string,
    order_ts timestamp,

    customer_sk string,
    product_sk string,
    
    qty int,
    list_price double,
    discount double,

    payment_mode string,
    store_num string,

    file_date string,
    source_system string,
    ingest_ts timestamp
)
 using delta;

In [0]:
%sql
--- reject orders with no lookup data
create table if not exists `spark-databricks-project-catalog`.silver.reject_orders
as (
    select *,
        cast('dummy' as varchar(100)) as reject_reason 
     from `spark-databricks-project-catalog`.silver.orders_unified
     where 1 = 2
    );

delete from `spark-databricks-project-catalog`.silver.reject_orders
 where file_date = :file_run_date;
 --using (:) -> as its a variable

insert into `spark-databricks-project-catalog`.silver.reject_orders -- rejected records would be inserted into this

select o.*,
       case when c.customer_sk is null then 'customer not found'
            else 'product not found'
            end 
            as reject_reason
 from `spark-databricks-project-catalog`.silver.orders_unified o
 
 -- ✅ customer lookup
 left join `spark-databricks-project-catalog`.gold.customers c
 on o.customer_id = c.offline_customer_id -- Take the customer ID from the order and check whether    that                                         customer exists in the Gold Customer table.
 or o.customer_id = c.online_customer_id

-- ✅ product SCD2 lookup
 left join `spark-databricks-project-catalog`.gold.products p
  on o.sku_id = p.sku_id 
  and to_date(o.file_date, 'yyyyMMdd') between p.effective_from and p.effective_to
where o.file_date = :file_run_date and 
(c.customer_sk is null OR p.product_sk is null);

--Take the SKU from the order and find the product version that was valid on that order's date.

In [0]:
%sql
delete from `spark-databricks-project-catalog`.gold.fact_orders where 
             file_date = :file_run_date;
insert into `spark-databricks-project-catalog`.gold.fact_orders
select 
       o.order_id,
       o.order_ts,
       c.customer_sk,
       p.product_sk,
       o.qty,
       o.list_price,
       o.discount,
       o.payment_mode,
       o.store_num,
       o.file_date,
       o.source_system,
        o.ingestion_time
 from `spark-databricks-project-catalog`.silver.orders_unified o 

-- ✅ customer lookup
 inner join `spark-databricks-project-catalog`.gold.customers c 
 on o.customer_id = c.online_customer_id
 or o.customer_id = c.offline_customer_id

-- ✅ product SCD2 lookup
 inner join `spark-databricks-project-catalog`.gold.products p 
 on o.sku_id = p.sku_id
 and to_date(o.file_date, 'yyyyMMdd') between p.effective_from and p.effective_to
 where o.file_date = :file_run_date;      
 --not only always latest data, but whenever catching the data, the record should've been active
 -- based on file date, we've to get the effective surrogate key